# Guiding in Prompts

In [1]:
#from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI
import os
from load_dotenv import load_dotenv

load_dotenv() #This function will load everything which is saved in our .env file and will make them available
              # in the os.environ dictionary (env variables)

if os.environ.get("OPENAI_API_KEY"):
    print("Bro, API KEY Variable Exists")
else:
    raise ValueError("OPENAI_API_KEY not found")

Bro, API KEY Variable Exists


In [2]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI

llm_openai = ChatOpenAI(model ='gpt-5-mini', temperature=0)

In [ ]:
from langchain_core.prompts import PromptTemplate


result = llm_openai.invoke("Tell me a joke. Generate the output in Key-Value pair format with the following keys: setup, punchline")

result.content  # Here we can see that we can guide LLM in the prompt.

'setup: "Why don\'t scientists trust atoms?"\npunchline: "Because they make up everything."'

 But it is simple when we have very simple prompts, but when we have let's say a very detailed prompt and dynamic prompt and when we are running complex workflows at that time we cannot just guide the LLM to generate the O/P in a specific format.

 Then we have to follow the structured approch. That called is Pydantic.

# Using Pydantic Models

In [6]:
from pydantic import BaseModel

class llm_schema(BaseModel):
    setup: str
    punchline: str

In [9]:
obj = llm_schema(**{"setup": "some setup", "punchline": "Some Punchline"})
obj

llm_schema(setup='some setup', punchline='Some Punchline')

In [10]:
obj = llm_schema(**{"setup": "some setup", "punchline": "Some Punchline"})
obj.setup

'some setup'

above we just get the setup, what is the advantage of this? this is a kind of Data validation library, if I pass some wrong thing it will throw an error called validation error

In [ ]:
obj = llm_schema(**{"ketchup": "some setup", "punchline": "Some Punchline"})
obj

ValidationError: 1 validation error for llm_schema
setup
  Field required [type=missing, input_value={'ketchup': 'some setup',...line': 'Some Punchline'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

it can only accept "setup" and "Punchline". if we will not provide that things, it will not allow us to create an object.

That means if LLM generates an output and it does not follow our schema, it will not allow LLM to generate the O/P and it will protect us to just deliver that pseudo schema or broken schema to our downstream workflows.

It also does data type check as well. For ex, if we pass wrong data type entry then also it will fail.

In [12]:
obj = llm_schema(**{"setup": 123, "punchline": "Some Punchline"})
obj

ValidationError: 1 validation error for llm_schema
setup
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

we define the data type for setup as string, but the input is integer, it will throw an error.

Till here DATA VALIDATION has been done. Now what about DATA PARSING?

llm_schema(**{"setup": "Some Setup", "punchline": "Some Punchline"})

This setup is called data parsing. When you are parsing the value

In [ ]:
obj = llm_schema(**{"setup": "some setup", "punchline": "Some Punchline"})
obj

What we did, we simply passed that particular class and we passed our arguments: {"setup": "some setup", "punchline": "Some Punchline"}

and then only we got our object.

In [ ]:
llm_structured_output = llm_openai.with_structured_output(llm_schema)

Now if we use above model (llm_structured_output), this will generate every time the output in this particular structure

class llm_schema(BaseModel):
    setup: str
    punchline: str

and we will not even tell LLM that generate the final output in the structure.

One PRO TIP:

whenever you are defining your class, you should always define a kind of description. Why? 

Because LLM is not magic, LLM will use your information as context.


In [ ]:
from pydantic import BaseModel, Field

class llm_schema(BaseModel):
    setup: str = Field(description="The setup for the joke")
    punchline: str = Field(description="The punchline for the joke")

LLM are all about context. More and precise context we will provide better applications we will be able to build.

In [15]:
llm_structured_output = llm_openai.with_structured_output(llm_schema)

llm_structured_output.invoke("Tell me a joke")

llm_schema(setup='Why did the scarecrow win an award?', punchline='Because he was outstanding in his field.')

Above we got the response in the form of llm_schema (which is the class). Then setup and puchline

This is pydantic class object.

In [ ]:
llm_structured_output = llm_openai.with_structured_output(llm_schema)

result = llm_structured_output.invoke("Tell me a joke")

type(result)  # See the result, this is an llm_schema object.

__main__.llm_schema

# Using TypeDict

In [17]:
from typing import TypedDict

class llm_schema_td(TypedDict):
    setup: str
    punchline: str

In [19]:
obj = llm_schema_td({"setup": "some setup", "punchline": "some punchline"})
obj

{'setup': 'some setup', 'punchline': 'some punchline'}

above is not an object, it is simple dictionary

In [20]:
obj = llm_schema_td({"ketchup": "some setup", "punchline": "some punchline"})
obj

{'ketchup': 'some setup', 'punchline': 'some punchline'}

See the difference, in pydantic we would get an error, but here it throws the result with ketchup. 

Than means we made a mistake but still it will not fail, why??

it will create the output using our mistake and it will simply giving you an error only for the typing part.

basically these are for the type hinting. 

In [22]:
llm_structured_typed_dict = llm_openai.with_structured_output(llm_schema_td)

result = llm_structured_typed_dict.invoke("Tell me a joke")

result

{'setup': "Why don't scientists trust atoms?",
 'punchline': 'Because they make up everything.'}

Here we can see we will get the answer in the form of dictionary but not in the form of Pydantic object.

What is the advantage??

If let's say your workflows are not very errorprone and you are not very much dependant on the same keys, then we can simply use typeddict for just as a context.

but when your workflows are very strict and you do not want them to create any random key on its own, you can use pydantic.

So whenever you just want to provide some context to LLM then you can simply use TypedDict.
